# Assignment 2 - Elementary Probability and Information Theory 
# Boise State University NLP - Dr. Kennington

### Instructions and Hints:

* This notebook loads some data into a `pandas` dataframe, then does a small amount of preprocessing. Make sure your data can load by stepping through all of the cells up until question 1. 
* Most of the questions require you to write some code. In many cases, you will write some kind of probability function like we did in class using the data. 
* Some of the questions only require you to write answers, so be sure to change the cell type to markdown or raw text
* Don't worry about normalizing the text this time (e.g., lowercase, etc.). Just focus on probabilies. 
* Most questions can be answered in a single cell, but you can make as many additional cells as you need. 
* Follow the instructions on the corresponding assignment Trello card for submitting your assignment. 

In [1]:
pip install pandas

In [3]:
pip install pyarrow


   ---------------------------------------- 0.0/24.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/24.9 MB 1.7 MB/s eta 0:00:16
   ---------------------------------------- 0.1/24.9 MB 1.6 MB/s eta 0:00:16
   ---------------------------------------- 0.2/24.9 MB 1.5 MB/s eta 0:00:17
    --------------------------------------- 0.4/24.9 MB 1.8 MB/s eta 0:00:14
    --------------------------------------- 0.6/24.9 MB 1.9 MB/s eta 0:00:13
   - -------------------------------------- 0.7/24.9 MB 1.9 MB/s eta 0:00:13
   - -------------------------------------- 0.8/24.9 MB 2.1 MB/s eta 0:00:12
   - -------------------------------------- 0.8/24.9 MB 2.1 MB/s eta 0:00:12
   - -------------------------------------- 1.0/24.9 MB 1.9 MB/s eta 0:00:13
   - -------------------------------------- 1.1/24.9 MB 1.9 MB/s eta 0:00:13
   - -------------------------------------- 1.2/24.9 MB 1.8 MB/s eta 0:00:14
   - -------------------------------------- 1.2/24.9 MB 1.7 MB/s eta 0:00:14
   --

In [2]:
import pandas as pd 

data = pd.read_csv('C:/Users/Sharadha Kasi/pnp-train.txt',delimiter='\t',encoding='latin-1', # utf8 encoding didn't work for this
                  names=['type','name']) # supply the column names for the dataframe

# this next line creates a new column with the lower-cased first word
data['first_word'] = data['name'].map(lambda x: x.lower().split()[0])

In [3]:
data[:10]

,type,name,first_word
0,drug,Dilotab,dilotab
1,movie,Beastie Boys: Live in Glasgow,beastie
2,person,Michelle Ford-Eriksson,michelle
3,place,Ramsbury,ramsbury
4,place,Market Bosworth,market
5,drug,Cyanide Antidote Package,cyanide
6,person,Bill Johnson,bill
7,place,Ettalong,ettalong
8,movie,The Suicide Club,the
9,place,Pézenas,pézenas


In [3]:
data.describe()

,type,name,first_word
count,21001,21001,21001
unique,5,20992,13703
top,movie,Epinal,the
freq,6262,2,635


## 1. Write a probability function/distribution $P(T)$ over the types. 

Hints:

* The Counter library might be useful: `from collections import Counter`
* Write a function `def P(T='')` that returns the probability of the specific value for T
* You can access the types from the dataframe by calling `data['type']`

In [7]:
from collections import Counter

def P(T=''):
    frequency_type = Counter(data['type'])
    total_possible_outcomes = len(data)
    probability_T = frequency_type[T] / total_possible_outcomes #if T in frequency_type else 0
    return probability_T

## 2. What is `P(T='movie')` ?

In [9]:
P(T='movie')

0.29817627732012764

## 3. Show that your probability distribution sums to one.

In [10]:
distinct_types = data['type'].unique()
probability_distribution_sum = sum(P(T) for T in distinct_types)
print(probability_distribution_sum)

1.0


## 4. Write a joint distribution using the type and the first word of the name

Hints:

* The function is $P2(T,W_1)$
* You will need to count up types AND the first words, for example: ('person','bill)
* Using the [itertools.product](https://docs.python.org/2/library/itertools.html#itertools.product) function was useful for me here

In [5]:
import itertools
from collections import Counter

def P2(T='', W1=''):
    frequency_type = Counter(data['type'])
    frequency_first_word = Counter(data['first_word'])
    total_possible_outcomes = len(data)
    joint_distribution = 0
    possibilities = list(itertools.product(frequency_type.keys(), frequency_first_word.keys()))
    frequency_possibilities = sum(1 for row in data.itertuples() if (row.type, row.first_word) == (T, W1))
    joint_distribution = frequency_possibilities / total_possible_outcomes
    return joint_distribution

## 5. What is P2(T='person', W1='bill')? What about P2(T='movie',W1='the')?

In [8]:
P2(T='person', W1='bill')

0.00047616780153326033

In [9]:
P2(T='movie', W1='the')

0.02747488214846912

## 6. Show that your probability distribution P(T,W1) sums to one.

In [6]:
distinct_types = data['type'].unique()
distinct_first_words = data['first_word'].unique()
total_joint_distribution = sum(P2(T, W1) for T in distinct_types for W1 in distinct_first_words)
print(total_joint_distribution)

0.9999999999997995


## 7. Make a new function Q(T) from marginalizing over P(T,W1) and make sure that Q(T) sums to one.

Hints:

* Your Q function will call P(T,W1)
* Your check for the sum to one should be the same answer as Question 3, only it calls Q instead of P.

In [21]:
def Q(T=''):
    marginal_probability = sum(P2(T, W1) for W1 in data['first_word'].unique())
    return marginal_probability

In [22]:
Q('movie')

0.29817627732011875

In [23]:
distinct_types = data['type'].unique()
total_marginal_probability = sum(Q(T) for T in distinct_types)
print(total_marginal_probability)

1.0000000000000266


## 8. What is the KL Divergence of your Q function and your P function for Question 1?

* Even if you know the answer, you still need to write code that computes it.

In [ ]:
import math

distinct_types = data['type'].unique()
kl_divergence = sum(P(T) * math.log(P(T) / Q(T)) for T in distinct_types)
print(kl_divergence)

## 9. Convert from P(T,W1) to P(W1|T) 

Hints:

* Just write a comment cell, no code this time. 
* Note that $P(T,W1) = P(W1,T)$

(try to use markdown math formating, answer in this cell)

To convert from P(T,W1​) to P(W1​∣T), using Bayes’ Theorem:
P(W1​∣T)=(P(T∣W1​).P(W1​)​)/P(T)
Since P(T,W1​)=P(W1​,T) ------->  P(W1​∣T)=(P(T∣W1​).P(W1​))/(P(T∣W1​).P(W1​) + P(T∣W2​).P(W2​))​
Here W2​ is the complement of W1​. This allows us to calculate the probability of W1​ given T.

## 10. Write a function `Pwt` (that calls the functions you already have) to compute $P(W_1|T)$.

* This will be something like the multiplication rule, but you may need to change something

In [10]:
def Pwt(T='', W1=''):
    joint_probability = P2(T, W1)
    marginal_probability = P(T)
    epsilon = 1e-10
    marginal_probability = max(marginal_probability, epsilon)
    # Conditional probability -- P(W1|T)
    conditional_probability = joint_probability / marginal_probability
    return conditional_probability

## 11. What is P(W1='the'|T='movie')?

In [11]:
Pwt(W1='the',T='movie')

0.09214308527626956

## 12. Use Baye's rule to convert from P(W1|T) to P(T|W1). Write a function Ptw to reflect this. 

Hints:

* Call your other functions.
* You may need to write a function for P(W1) and you may need a new counter for `data['first_word']`

In [12]:
from collections import Counter

def Ptw(T='', W1=''):
    conditional_probability_W1_given_T = Pwt(T, W1)
    prior_probability_T = P(T)

    # Calculate probability of W1
    frequency_word = Counter(data['first_word'])
    total_words = len(data['first_word'])
    probability_W1 = frequency_word[W1] / total_words
    epsilon = 1e-10
    probability_W1 = max(probability_W1, epsilon)
    posterior_probability_T_given_W1 = (conditional_probability_W1_given_T * prior_probability_T) / probability_W1
    return posterior_probability_T_given_W1

## 13 
### What is P(T='movie'|W1='the')? 
### What about P(T='person'|W1='the')?
### What about P(T='drug'|W1='the')?
### What about P(T='place'|W1='the')
### What about P(T='company'|W1='the')

In [13]:
Ptw(T='movie',W1='the')

0.9086614173228347

In [14]:
Ptw(T='person',W1='the')

0.0

In [15]:
Ptw(T='drug',W1='the')

0.0

In [16]:
Ptw(T='place',W1='the')

0.0015748031496062992

In [17]:
Ptw(T='company',W1='the')

0.08976377952755905

## 14 Given this, if the word 'the' is found in a name, what is the most likely type?

In [18]:
distinct_types = data['type'].unique()
posterior_probabilities = {T: Ptw(T, 'the') for T in distinct_types}
most_likely_type = max(posterior_probabilities, key=posterior_probabilities.get)
print(most_likely_type)

movie


## 15. Is Ptw(T='movie'|W1='the') the same as Pwt(W1='the'|T='movie') the same? Why or why not?

In [19]:
Ptw(T='movie',W1='the')

0.9086614173228347

In [20]:
Pwt(W1='the', T='movie')

0.09214308527626956

(why or why not? answer here)

No, Ptw(T='movie'|W1='the') and Pwt(W1='the'|T='movie') are not the same. Here, the main difference arises due to the difference in conditional probability which is termed as 'given' information. In the first case we are calculationg the posterior probability of the type 'movie' given the presence of the word 'the'. In the second case we are calculationg the conditional probability of the word 'the' given the presence of the type 'movie'. So both the probabilities are related but they are not the same. Main difference is therefore present in the 'given' information

## 16. Do you think modeling Ptw(T|W1) would be better with a continuous function like a Gaussian? Why or why not?

- Answer in a markdown cell


Modeling Ptw(T|W1) with a continuous function like gaussian could be beneficial at certain times but this depebds on the data and probability distribution that we may use. Given a condition that we use probabilities that are bounded between 0 and 1, it may not be an ideal choice. This is because, gaussian distibutions assume values over the real line, so in this case discrete models might be suitable. So, if the given data set has non-gaussian characteristics it should better to avoid it, else it can be used.